In [1]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


cwd: <repo>\fine-tuning


# Lab 01 · Supervised Fine-Tuning — teach and measure policy facts

Train `gpt-4.1-mini` on source-grounded Acme Health Q&A pairs, then compare the base and tuned deployments on held-out paraphrases. Deterministic fact checks separate real improvement from parity, regression, or a confident but unsupported answer.

---
## Step 1 — Configuration & client

In [2]:
import os, json, time, hashlib, re, requests
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import AzureCliCredential, get_bearer_token_provider

load_dotenv()

AZURE_OPENAI_ENDPOINT     = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION  = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_MODEL                = os.environ.get('BASE_MODEL', 'gpt-4.1-mini-2025-04-14')
BASE_DEPLOYMENT           = os.environ.get('BASE_DEPLOYMENT', 'gpt-4.1-mini')
SUBSCRIPTION_ID           = os.environ['AZURE_SUBSCRIPTION_ID']
RESOURCE_GROUP            = os.environ['AZURE_RESOURCE_GROUP']
RESOURCE_NAME             = os.environ['AZURE_RESOURCE_NAME']
TENANT_ID                 = os.environ['AZURE_TENANT_ID']

SYSTEM_PROMPT = (
    'You are the Acme Health AI Assistant. Answer member '
    "questions accurately according to Acme Health's official policies, "
    'pharmacy procedures, plan benefits, and the My Health Online portal.'
)

_cred = AzureCliCredential(tenant_id=TENANT_ID)
token_provider = get_bearer_token_provider(
    _cred,
    'https://cognitiveservices.azure.com/.default',
)
client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider=token_provider,
    api_version=AZURE_OPENAI_API_VERSION,
)

print(f'Endpoint    : {AZURE_OPENAI_ENDPOINT}')
print(f'Tenant      : {TENANT_ID}')
print(f'Base model  : {BASE_MODEL}')
print(f'Base deploy : {BASE_DEPLOYMENT}')

Endpoint    : https://<your-resource>.cognitiveservices.azure.com/
Tenant      : <your-tenant-id>
Base model  : gpt-4.1-mini-2025-04-14
Base deploy : gpt-4.1-mini


---
## Step 2 — Establish the base-model baseline

These held-out paraphrases test whether each model returns the required policy facts. The base model may pass some cases; only measured score differences count as SFT improvement.

In [7]:
def pm_hours(text):
    return set(re.findall(r'\b(1[0-2]|[1-9])(?::\d{2})?\s*p\.?m\.?(?=\s|$|[,.;])', text, re.I))

def cutoff_checks():
    return {
        '2:00 PM cutoff': lambda text: '2' in pm_hours(text),
        'no conflicting PM cutoff': lambda text: pm_hours(text) <= {'2'},
        'next business day': lambda text: 'next business day' in text.casefold(),
    }

def mail_order_checks():
    return {
        '$20 copay': lambda text: bool(re.search(r'\$\s*20(?:\.00)?\b', text)),
        'no $15 copay': lambda text: not bool(re.search(r'\$\s*15(?:\.00)?\b', text)),
    }

def formulary_checks():
    return {
        '60 days': lambda text: bool(re.search(r'\b60\s+days?\b', text, re.I)),
        'email': lambda text: bool(re.search(r'\be-?mail', text, re.I)),
        'affected members': lambda text: 'affected member' in text.casefold(),
    }

EVALUATION_CASES = [
    {
        'policy': 'Refill cutoff',
        'question': 'What is the same-day refill request cutoff at ACME pharmacies, and what happens after it?',
        'expected': 'Before 2:00 PM Pacific for same-day filling; after 2:00 PM, the next business day.',
        'checks': cutoff_checks(),
    },
    {
        'policy': 'Refill cutoff',
        'question': 'I sent a refill at 1:45 PM Pacific. Should ACME fill it today, and what if I send it at 2:15 PM?',
        'expected': 'Before 2:00 PM Pacific is filled the same business day; after 2:00 PM is filled the next business day.',
        'checks': cutoff_checks(),
    },
    {
        'policy': 'Refill cutoff',
        'question': 'State the Pacific-time deadline for same-business-day ACME refill processing and the handling after that deadline.',
        'expected': 'The deadline is 2:00 PM Pacific; later requests are filled the next business day.',
        'checks': cutoff_checks(),
    },
    {
        'policy': 'Mail-order copay',
        'question': 'How much does a 90-day mail-order refill of a Tier 1 generic cost on Acme Health?',
        'expected': '$20 copay.',
        'checks': mail_order_checks(),
    },
    {
        'policy': 'Mail-order copay',
        'question': 'For a preferred generic, what should a member owe when ordering a three-month supply by mail?',
        'expected': '$20 copay for a 90-day Tier 1 preferred generic.',
        'checks': mail_order_checks(),
    },
    {
        'policy': 'Mail-order copay',
        'question': 'Quote the Tier 1 copay for the 90-day mail-service benefit.',
        'expected': '$20 copay.',
        'checks': mail_order_checks(),
    },
    {
        'policy': 'Formulary notice',
        'question': 'How and when does ACME notify members affected by a major formulary change?',
        'expected': 'ACME emails affected members 60 days in advance.',
        'checks': formulary_checks(),
    },
    {
        'policy': 'Formulary notice',
        'question': 'Describe the channel and lead time used to alert members whose drug coverage is affected by a major formulary revision.',
        'expected': 'ACME emails affected members 60 days before the revision.',
        'checks': formulary_checks(),
    },
    {
        'policy': 'Formulary notice',
        'question': 'A major drug-list update affects me. What advance notice should I expect from ACME?',
        'expected': 'An email 60 days in advance to affected members.',
        'checks': formulary_checks(),
    },
]

TEST_QUESTIONS = [case['question'] for case in EVALUATION_CASES]
CORRECT_ANSWERS = [case['expected'] for case in EVALUATION_CASES]

def ask_model(deployment, questions):
    answers = []
    for question in questions:
        response = client.chat.completions.create(
            model=deployment,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': question},
            ],
            temperature=0.0,
            max_tokens=220,
        )
        answers.append(response.choices[0].message.content.strip())
    return answers

print('BEFORE Fine-Tuning - Base Model')
print('=' * 75)
base_answers = ask_model(BASE_DEPLOYMENT, TEST_QUESTIONS)
for index, (question, answer) in enumerate(zip(TEST_QUESTIONS, base_answers), 1):
    print(f'Q{index}: {question}')
    print(f'A{index}: {answer}\n')

BEFORE Fine-Tuning - Base Model
Q1: What is the same-day refill request cutoff at ACME pharmacies, and what happens after it?
A1: At Acme Health pharmacies, the same-day refill request cutoff is 2:00 PM. If you submit your refill request after 2:00 PM, your prescription will be processed the next business day. This means you may need to wait longer to pick up your medication if you request a refill after the cutoff time.

Q2: I sent a refill at 1:45 PM Pacific. Should ACME fill it today, and what if I send it at 2:15 PM?
A2: Acme Health's pharmacy typically processes refill requests received before 2:00 PM Pacific time on the same business day. Since you sent your refill at 1:45 PM Pacific, it should be filled today.

If you send a refill request at 2:15 PM Pacific, it will likely be processed on the next business day, as it is after the 2:00 PM cutoff time.

If you have any urgent needs, please contact the pharmacy directly for assistance.

Q3: State the Pacific-time deadline for same

---
## Step 3 — Validate the rebuilt datasets

Load plain UTF-8 JSONL, confirm the three required facts are in training, and prove that validation and held-out prompts do not leak into training.

In [12]:
TRAIN_FILE = Path('data/acme_training.jsonl')
VALID_FILE = Path('data/acme_validation.jsonl')
N_EPOCHS = 1
TRAINING_TYPE = 'GlobalStandard'
MIN_TRAIN_EXAMPLES_PER_FACT = 6

with open(TRAIN_FILE, 'r', encoding='utf-8-sig') as file:
    train_data = [json.loads(line) for line in file if line.strip()]
with open(VALID_FILE, 'r', encoding='utf-8-sig') as file:
    valid_data = [json.loads(line) for line in file if line.strip()]

def messages_for_role(row, role):
    return [message['content'] for message in row['messages'] if message['role'] == role]

def normalized_user_questions(rows):
    return [
        question.strip().casefold()
        for row in rows
        for question in messages_for_role(row, 'user')
    ]

def assistant_response(row):
    return '\n'.join(messages_for_role(row, 'assistant')).casefold()

def fact_coverage_counts(rows):
    answers = [assistant_response(row) for row in rows]
    return {
        '2:00 PM and next business day': sum(
            '2:00 pm' in answer and 'next business day' in answer
            for answer in answers
        ),
        '$20 Tier 1 mail order': sum(
            '$20' in answer
            and 'tier 1' in answer
            and ('90-day' in answer or '90 day' in answer)
            and 'mail' in answer
            for answer in answers
        ),
        '60-day emailed notice': sum(
            '60 days' in answer
            and 'email' in answer
            and 'affected member' in answer
            for answer in answers
        ),
    }

train_question_list = normalized_user_questions(train_data)
valid_question_list = normalized_user_questions(valid_data)
train_questions = set(train_question_list)
valid_questions = set(valid_question_list)
evaluation_questions = {question.casefold() for question in TEST_QUESTIONS}

assert len(train_question_list) == len(train_questions), 'Duplicate training prompts detected.'
assert len(valid_question_list) == len(valid_questions), 'Duplicate validation prompts detected.'
assert not train_questions & valid_questions, 'Train/validation prompt leakage detected.'
assert not evaluation_questions & (train_questions | valid_questions), 'Evaluation prompt leakage detected.'

training_fact_counts = fact_coverage_counts(train_data)
validation_fact_counts = fact_coverage_counts(valid_data)
assert all(count >= MIN_TRAIN_EXAMPLES_PER_FACT for count in training_fact_counts.values()), (
    f'Each required fact needs at least {MIN_TRAIN_EXAMPLES_PER_FACT} training examples: '
    f'{training_fact_counts}'
)
assert all(count >= 1 for count in validation_fact_counts.values()), (
    f'Each required fact needs validation coverage: {validation_fact_counts}'
)

dataset_sha256 = hashlib.sha256(
    TRAIN_FILE.read_bytes() + b'\0' + VALID_FILE.read_bytes()
).hexdigest()
run_config = {
    'dataset_sha256': dataset_sha256,
    'base_model': BASE_MODEL,
    'n_epochs': N_EPOCHS,
    'seed': 42,
    'training_type': TRAINING_TYPE,
}
run_sha256 = hashlib.sha256(
    json.dumps(run_config, sort_keys=True).encode('utf-8')
).hexdigest()
FT_DEPLOYMENT_NAME = os.environ.get('FT_DEPLOYMENT_NAME') or f'acme-sft-{run_sha256[:8]}'

print(f'Training      : {len(train_data)} examples from {TRAIN_FILE}')
print(f'Validation    : {len(valid_data)} examples from {VALID_FILE}')
print(f'Train coverage: {training_fact_counts}')
print(f'Valid coverage: {validation_fact_counts}')
print(f'Dataset SHA   : {dataset_sha256}')
print(f'Run SHA       : {run_sha256}')
print(f'Epochs        : {N_EPOCHS}')
print(f'Deployment    : {FT_DEPLOYMENT_NAME}')

Training      : 64 examples from data\acme_training.jsonl
Validation    : 11 examples from data\acme_validation.jsonl
Train coverage: {'2:00 PM and next business day': 6, '$20 Tier 1 mail order': 6, '60-day emailed notice': 6}
Valid coverage: {'2:00 PM and next business day': 1, '$20 Tier 1 mail order': 1, '60-day emailed notice': 1}
Dataset SHA   : 76123a7162511e1758dd338339955f3f1980fb562838d4ec49b7252f669c88a0
Run SHA       : 2be1642d0786dff3b6d3efd99562cc73f87785cd3323324afa57df4d2ac0279e
Epochs        : 1
Deployment    : acme-sft-2be1642d


---
## Step 4 — Upload files to Azure OpenAI

In [13]:
from openai import NotFoundError

_job_state_file = Path('.sft_job_state.json')
job_id = None

if _job_state_file.exists():
    saved_state = json.loads(_job_state_file.read_text(encoding='utf-8'))
    if saved_state.get('run_sha256') == run_sha256:
        try:
            saved_job = client.fine_tuning.jobs.retrieve(saved_state['job_id'])
            job_id = saved_job.id
            training_file_id = saved_job.training_file
            validation_file_id = saved_job.validation_file
            print(f'Reusing matching run: {job_id} ({saved_job.status})')
        except NotFoundError:
            print('Matching run marker returned 404; new files will be uploaded.')
    else:
        print('Dataset or training configuration changed; the previous job will not be reused.')

if job_id is None:
    print('Uploading rebuilt training file...')
    with open(TRAIN_FILE, 'rb') as file:
        train_resp = client.files.create(file=file, purpose='fine-tune')
    training_file_id = train_resp.id
    print(f'  -> {training_file_id} ({train_resp.status})')

    print('Uploading rebuilt validation file...')
    with open(VALID_FILE, 'rb') as file:
        valid_resp = client.files.create(file=file, purpose='fine-tune')
    validation_file_id = valid_resp.id
    print(f'  -> {validation_file_id} ({valid_resp.status})')

Dataset or training configuration changed; the previous job will not be reused.
Uploading rebuilt training file...
  -> file-e13c0aa3dc314a6a8909a02f15a1bce7 (pending)
Uploading rebuilt validation file...
  -> file-61a4d81fb9da4f398c379a0cec4f47d6 (pending)


---
## Step 5 — Submit the SFT job

Training tiers: `Standard` (regional, ~$1.70/hr) · `GlobalStandard` (cheaper,
globally distributed) · `Developer` (free, preemptible, deleted after 24h).

In [14]:
from datetime import datetime, timedelta, timezone

if job_id is None:
    file_deadline = datetime.now(timezone.utc) + timedelta(minutes=5)
    for file_id, label in (
        (training_file_id, 'training'),
        (validation_file_id, 'validation'),
    ):
        while datetime.now(timezone.utc) < file_deadline:
            uploaded_file = client.files.retrieve(file_id)
            print(f'{label.capitalize()} file: {uploaded_file.status}')
            if uploaded_file.status == 'processed':
                break
            if uploaded_file.status == 'error':
                raise RuntimeError(f'{label.capitalize()} file import failed: {uploaded_file.status_details}')
            time.sleep(5)
        else:
            raise TimeoutError(f'{label.capitalize()} file was not processed within 5 minutes.')

    print('Submitting a new SFT job for this run fingerprint...')
    job = client.fine_tuning.jobs.create(
        training_file=training_file_id,
        validation_file=validation_file_id,
        model=BASE_MODEL,
        suffix=f'acme-{run_sha256[:8]}',
        seed=42,
        hyperparameters={'n_epochs': N_EPOCHS},
        extra_body={'trainingType': TRAINING_TYPE},
    )
    job_id = job.id
    _job_state_file.write_text(
        json.dumps(
            {
                'job_id': job_id,
                'run_sha256': run_sha256,
                **run_config,
                'training_file_id': training_file_id,
                'validation_file_id': validation_file_id,
                'deployment_name': FT_DEPLOYMENT_NAME,
            },
            indent=2,
        ),
        encoding='utf-8',
    )

job_status = client.fine_tuning.jobs.retrieve(job_id)
print(f'Job ID     : {job_id}')
print(f'Status     : {job_status.status}')
print(f'Base model : {job_status.model}')
print(f'Run        : {run_sha256[:12]}')
print(f'Epochs     : {N_EPOCHS}')

Training file: processed
Validation file: processed
Submitting a new SFT job for this run fingerprint...
Job ID     : ftjob-94c3794462a1487a97072c50d78d55c9
Status     : pending
Base model : gpt-4.1-mini-2025-04-14
Run        : 2be1642d0786
Epochs     : 1


---
## Step 6 — Check status without waiting

Run the next cell once for the current service status. If the job is still `pending`, `validating_files`, or `running`, the cell returns immediately; re-run it later. A successful job initializes the model and file IDs used by the remaining steps.

In [15]:
from datetime import datetime, timezone

if 'client' not in globals():
    load_dotenv()
    TENANT_ID = os.environ['AZURE_TENANT_ID']
    _cred = AzureCliCredential(tenant_id=TENANT_ID)
    token_provider = get_bearer_token_provider(
        _cred,
        'https://cognitiveservices.azure.com/.default',
    )
    client = AzureOpenAI(
        azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
        azure_ad_token_provider=token_provider,
        api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
    )

saved_state = json.loads(Path('.sft_job_state.json').read_text(encoding='utf-8'))
job_id = saved_state['job_id']
job_status = client.fine_tuning.jobs.retrieve(job_id)
created_at = datetime.fromtimestamp(job_status.created_at, tz=timezone.utc)
elapsed = datetime.now(timezone.utc) - created_at

print(f'Job     : {job_id}')
print(f'Status  : {job_status.status}')
print(f'Created : {created_at:%Y-%m-%d %H:%M:%S} UTC')
print(f'Elapsed : {elapsed}')

recent_events = list(client.fine_tuning.jobs.list_events(
    fine_tuning_job_id=job_id,
    limit=10,
))
if recent_events:
    print('\nRecent service events:')
    for event in reversed(recent_events[-5:]):
        event_time = datetime.fromtimestamp(event.created_at, tz=timezone.utc)
        print(f'  {event_time:%H:%M:%S}  {event.message}')

sibling_jobs = [
    candidate for candidate in client.fine_tuning.jobs.list(limit=10)
    if candidate.id != job_id
]
if sibling_jobs:
    print('\nRecent sibling jobs:')
    for candidate in sibling_jobs[:5]:
        candidate_time = datetime.fromtimestamp(candidate.created_at, tz=timezone.utc)
        print(f'  {candidate.id}  {candidate.status}  created {candidate_time:%Y-%m-%d %H:%M:%S} UTC')

if job_status.status == 'succeeded':
    fine_tuned_model = job_status.fine_tuned_model
    training_file_id = job_status.training_file
    validation_file_id = job_status.validation_file
    print(f'\nFine-tuned model   : {fine_tuned_model}')
    print(f'Training file ID  : {training_file_id}')
    print(f'Validation file ID: {validation_file_id}')
elif job_status.status in {'failed', 'cancelled'}:
    raise RuntimeError(f'Fine-tuning ended in {job_status.status}: {job_status.error}')
else:
    print('\nTraining remains server-side; this status check is complete.')

Job     : ftjob-94c3794462a1487a97072c50d78d55c9
Status  : pending
Created : 2026-09-03 17:59:12 UTC
Elapsed : 0:00:12.494276

Recent service events:
  17:59:12  Job enqueued. Waiting for jobs ahead to complete.

Recent sibling jobs:
  ftjob-e91dd10fc3dc40a38f83f2e3e175900a  succeeded  created 2026-08-18 16:40:03 UTC
  ftjob-b7ed11a11773437b9a8b991d2fb00cca  succeeded  created 2026-08-18 16:00:52 UTC
  ftjob-e9d104a5251345c39a3bbda541c3e2e8  succeeded  created 2026-08-18 15:47:20 UTC
  ftjob-9f4a52cfcbec4383b3897c4029c559e6  succeeded  created 2026-08-18 15:20:53 UTC

Training remains server-side; this status check is complete.


---
## Step 7 — Plot training metrics

Watch for `train_loss` ↓ and `full_valid_loss` ↓ moving together. Divergence
(train still falling but valid rising) = overfitting → use an earlier checkpoint.

In [17]:
# Resumable: re-import plotting deps if the kernel was restarted.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

final_job = client.fine_tuning.jobs.retrieve(job_id)
if final_job.result_files:
    rid = final_job.result_files[0]
    content = client.files.content(rid).read()
    Path('data/acme_sft_results.csv').write_bytes(content)
    df = pd.read_csv('data/acme_sft_results.csv')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle('ACME SFT Training Metrics', fontweight='bold')

    ax = axes[0]
    if 'train_loss' in df.columns:
        ax.plot(df['step'], df['train_loss'], label='Train Loss', linewidth=1.5)
    if 'valid_loss' in df.columns:
        m = df['valid_loss'].notna()
        ax.plot(df.loc[m, 'step'], df.loc[m, 'valid_loss'], label='Valid Loss', linewidth=1.5, linestyle='--')
    if 'full_valid_loss' in df.columns:
        m = df['full_valid_loss'].notna()
        ax.scatter(df.loc[m, 'step'], df.loc[m, 'full_valid_loss'], label='Full Valid Loss (epoch end)', color='red', zorder=5)
    ax.set_title('Loss')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    if 'train_mean_token_accuracy' in df.columns:
        ax.plot(df['step'], df['train_mean_token_accuracy'], label='Train Token Acc', linewidth=1.5)
    if 'full_valid_mean_token_accuracy' in df.columns:
        m = df['full_valid_mean_token_accuracy'].notna()
        ax.scatter(df.loc[m, 'step'], df.loc[m, 'full_valid_mean_token_accuracy'], label='Full Valid Acc (epoch end)', color='red', zorder=5)
    ax.set_title('Token Accuracy')
    ax.set_xlabel('Step'); ax.set_ylabel('Accuracy'); ax.legend(); ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('data/acme_sft_metrics.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No result file yet.')

No result file yet.


---
## Step 8 — Deploy the fine-tuned model (ARM control plane)

Fine-tuned models live in your resource for free, but you have to deploy
them to call them. Deployment costs ~$1.70/hour while live.

In [63]:
if 'fine_tuned_model' not in globals():
    job_status = client.fine_tuning.jobs.retrieve(job_id)
    if job_status.status != 'succeeded':
        raise RuntimeError(f'Job is not deployable: {job_status.status}')
    fine_tuned_model = job_status.fine_tuned_model

if 'FT_DEPLOYMENT_NAME' not in globals():
    saved_state = json.loads(Path('.sft_job_state.json').read_text(encoding='utf-8'))
    FT_DEPLOYMENT_NAME = saved_state['deployment_name']

management_token = _cred.get_token('https://management.azure.com/.default').token
deploy_url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}'
    f'/providers/Microsoft.CognitiveServices/accounts/{RESOURCE_NAME}'
    f'/deployments/{FT_DEPLOYMENT_NAME}'
)
response = requests.put(
    deploy_url,
    params={'api-version': '2024-10-01'},
    headers={
        'Authorization': f'Bearer {management_token}',
        'Content-Type': 'application/json',
    },
    json={
        'sku': {'name': 'GlobalStandard', 'capacity': 1},
        'properties': {
            'model': {
                'format': 'OpenAI',
                'name': fine_tuned_model,
                'version': '1',
            }
        },
    },
    timeout=60,
)
response.raise_for_status()
print(f'Deployment requested: {FT_DEPLOYMENT_NAME} (HTTP {response.status_code})')

Deployment requested: acme-sft-509ef015 (HTTP 201)


In [64]:
from datetime import datetime, timedelta, timezone

deadline = datetime.now(timezone.utc) + timedelta(minutes=15)
while datetime.now(timezone.utc) < deadline:
    response = requests.get(
        deploy_url,
        params={'api-version': '2024-10-01'},
        headers={'Authorization': f'Bearer {management_token}'},
        timeout=30,
    )
    response.raise_for_status()
    deployment_state = response.json().get('properties', {}).get('provisioningState', 'Unknown')
    print(f'[{time.strftime("%H:%M:%S")}] {deployment_state}')
    if deployment_state == 'Succeeded':
        break
    if deployment_state in {'Failed', 'Canceled'}:
        raise RuntimeError(json.dumps(response.json(), indent=2))
    time.sleep(20)
else:
    raise TimeoutError(f'Deployment {FT_DEPLOYMENT_NAME} was not ready within 15 minutes.')

[12:22:22] Creating
[12:22:44] Creating
[12:23:05] Creating
[12:23:27] Creating
[12:23:48] Creating
[12:24:10] Creating
[12:24:31] Creating
[12:24:53] Creating
[12:25:14] Creating
[12:25:36] Creating
[12:25:58] Creating
[12:26:19] Creating
[12:26:41] Creating
[12:27:02] Succeeded


---
## Step 9 — Run the held-out evaluation on the fine-tuned model

Use the same prompts and deterministic settings as the baseline. The next step grades required facts rather than relying on visual confidence.

In [65]:
if 'client' not in globals():
    load_dotenv()
    _cred = AzureCliCredential(tenant_id=os.environ['AZURE_TENANT_ID'])
    token_provider = get_bearer_token_provider(
        _cred,
        'https://cognitiveservices.azure.com/.default',
    )
    client = AzureOpenAI(
        azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
        azure_ad_token_provider=token_provider,
        api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
    )

if 'FT_DEPLOYMENT_NAME' not in globals():
    saved_state = json.loads(Path('.sft_job_state.json').read_text(encoding='utf-8'))
    FT_DEPLOYMENT_NAME = saved_state['deployment_name']

print('AFTER Fine-Tuning - SFT Model')
print('=' * 75)
ft_answers = ask_model(FT_DEPLOYMENT_NAME, TEST_QUESTIONS)
for index, (question, answer) in enumerate(zip(TEST_QUESTIONS, ft_answers), 1):
    print(f'Q{index}: {question}')
    print(f'A{index}: {answer}\n')

AFTER Fine-Tuning - SFT Model
Q1: What is the same-day refill request cutoff at ACME pharmacies, and what happens after it?
A1: Same-day refill requests must be made by 5:00 PM Pacific Time. Requests made after 5:00 PM are processed the next business day.

Q2: How much does a 90-day mail-order refill of a Tier 1 generic cost on Acme Health?
A2: A 90-day mail-order refill of a Tier 1 generic costs $15.

Q3: How and when does ACME notify members affected by a major formulary change?
A3: ACME notifies affected members at least 60 days before the change takes effect.



---
## Step 10 — Deterministic scorecard and release decision

Each case passes only when every required fact is present. “Improvement” means the fine-tuned model passes a case that the base model failed; a case both models pass is reported as parity.

In [ ]:
def grade_answer(answer, evaluation_case):
    check_results = {
        label: check(answer)
        for label, check in evaluation_case['checks'].items()
    }
    return all(check_results.values()), check_results

expected_count = len(EVALUATION_CASES)
if len(base_answers) != expected_count or len(ft_answers) != expected_count:
    raise RuntimeError(
        'Evaluation answer count mismatch. Re-run both model evaluation steps before grading: '
        f'expected={expected_count}, base={len(base_answers)}, fine_tuned={len(ft_answers)}'
    )

rows = []
for evaluation_case, base_answer, ft_answer in zip(EVALUATION_CASES, base_answers, ft_answers):
    base_pass, base_checks = grade_answer(base_answer, evaluation_case)
    ft_pass, ft_checks = grade_answer(ft_answer, evaluation_case)
    rows.append(
        {
            'Policy': evaluation_case['policy'],
            'Question': evaluation_case['question'],
            'Expected': evaluation_case['expected'],
            'Base pass': base_pass,
            'Fine-tuned pass': ft_pass,
            'Result': (
                'Improved' if ft_pass and not base_pass
                else 'Parity' if ft_pass == base_pass
                else 'Regressed'
            ),
            'Base missing': ', '.join(label for label, passed in base_checks.items() if not passed) or '-',
            'Fine-tuned missing': ', '.join(label for label, passed in ft_checks.items() if not passed) or '-',
        }
    )

scorecard = pd.DataFrame(rows)
base_score = int(scorecard['Base pass'].sum())
ft_score = int(scorecard['Fine-tuned pass'].sum())
release_pass = ft_score == expected_count and ft_score >= base_score

display(scorecard)
print(f'Base score       : {base_score}/{expected_count}')
print(f'Fine-tuned score : {ft_score}/{expected_count}')
print(f'Release gate     : {"PASS" if release_pass else "FAIL"}')

if release_pass:
    print('Conclusion: the fine-tuned model passed every required fact check.')
else:
    print('Conclusion: do not claim SFT success; inspect failed checks and iterate before release.')

,Question,Expected,Base pass,Fine-tuned pass,Result,Base missing,Fine-tuned missing
0,What is the same-day refill request cutoff at ...,Before 2:00 PM Pacific for same-day filling; a...,True,False,Regressed,-,2:00 PM cutoff
1,How much does a 90-day mail-order refill of a ...,$20 copay.,False,False,Parity,$20 copay,$20 copay
2,How and when does ACME notify members affected ...,ACME emails affected members 60 days in advance.,False,False,Parity,"email, affected members",email


Base score       : 1/3
Fine-tuned score : 0/3
Release gate     : FAIL
Conclusion: do not claim SFT success; inspect failed checks and iterate before release.


---
## Step 11 — Cleanup (delete the deployment, keep the model)

**Important:** deployments cost ~$1.70/hour even with zero traffic. The
fine-tuned model weights are stored for free until you delete them.

In [ ]:
load_dotenv()

if '_cred' not in globals():
    _cred = AzureCliCredential(tenant_id=os.environ['AZURE_TENANT_ID'])
if 'client' not in globals():
    token_provider = get_bearer_token_provider(
        _cred,
        'https://cognitiveservices.azure.com/.default',
    )
    client = AzureOpenAI(
        azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
        azure_ad_token_provider=token_provider,
        api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
    )

saved_state = json.loads(Path('.sft_job_state.json').read_text(encoding='utf-8'))
cleanup_deployment_name = saved_state['deployment_name']
training_file_id = saved_state.get('training_file_id')
validation_file_id = saved_state.get('validation_file_id')

for file_id, label in (
    (training_file_id, 'training'),
    (validation_file_id, 'validation'),
):
    if not file_id:
        print(f'No {label} file ID; skipped.')
        continue
    try:
        client.files.delete(file_id)
        print(f'Deleted {label} file: {file_id}')
    except Exception as error:
        print(f'Could not delete {label} file: {error}')

deploy_url = (
    f"https://management.azure.com/subscriptions/{os.environ['AZURE_SUBSCRIPTION_ID']}"
    f"/resourceGroups/{os.environ['AZURE_RESOURCE_GROUP']}"
    f"/providers/Microsoft.CognitiveServices/accounts/{os.environ['AZURE_RESOURCE_NAME']}"
    f'/deployments/{cleanup_deployment_name}'
)
management_token = _cred.get_token('https://management.azure.com/.default').token
response = requests.delete(
    deploy_url,
    params={'api-version': '2024-10-01'},
    headers={'Authorization': f'Bearer {management_token}'},
    timeout=60,
)
if response.status_code not in {200, 202, 204, 404}:
    response.raise_for_status()
print(f'Deployment delete ({cleanup_deployment_name}): HTTP {response.status_code}')

if 'fine_tuned_model' in globals():
    print(f'Fine-tuned model retained: {fine_tuned_model}')

Deleted training file: file-6432096c66fb496a98c270dc0eaa0114
Deleted validation file: file-22f9721b47284d628644a6798cfa3a23
Deployment delete (acme-sft-509ef015): HTTP 202
Fine-tuned model retained: gpt-4.1-mini-2025-04-14.ft-9f4a52cfcbec4383b3897c4029c559e6-acme-509ef015
